# Pipeline de ingeniería de características

**Proyecto #2 · Notebook 2 de 4** · Maestría · 4T 2026 · Product Management · Clase Cloud
**Autor:** Danny García

---

El notebook 02 dejó por escrito **qué** transformaciones hacen falta. Este las convierte en un
objeto `Pipeline` de scikit-learn y lo guarda en un archivo `.pkl`.

## Por qué un pipeline y no un script que transforme un CSV

Es la decisión más importante del proyecto, y conviene entender el motivo antes de ver el código.

La forma cómoda de trabajar sería transformar los datos una vez, guardar el resultado en un CSV y
entrenar sobre él. Tiene dos problemas graves:

1. **Fuga de información.** Si la media que usa el escalado, o la mediana que usa el imputador, o
   los percentiles del recorte se calculan sobre *todos* los datos, entonces el preprocesamiento
   **ya vio** las filas de validación. El error de validación sale optimista y el modelo decepciona
   en producción.
2. **Un modelo que no se puede usar.** Un CSV transformado no sirve para predecir datos nuevos.
   Habría que repetir a mano, en el momento de la inferencia, exactamente los mismos pasos y en el
   mismo orden. Cualquier diferencia produce predicciones silenciosamente equivocadas.

Un `Pipeline` resuelve ambos: **aprende solo del entrenamiento** y **sabe transformar datos
crudos**, así que el objeto que se guarda ya es capaz de ir de `test.csv` a predicciones.

## Qué entra y qué sale

| | |
|---|---|
| **Entra** | Tres columnas: `shop_id`, `item_id`, `date_block_num` |
| **Sale** | 29 columnas numéricas, sin nulos, escaladas, en `float32` |
| **Artefacto** | `models/pipeline_features.pkl` |

Que la entrada sean solo tres columnas es deliberado: `test.csv` trae `ID`, `shop_id` e `item_id`,
y el mes a predecir es la constante 34. Todo lo demás lo genera el pipeline.

## Índice

| # | Sección |
|---|---|
| 0 | [Configuración](#0) |
| 1 | [Los datos que el pipeline lleva dentro](#1) |
| 2 | [Los operadores que generan variables](#2) |
| 3 | [El ColumnTransformer: los seis procesos](#3) |
| 4 | [El pipeline ensamblado](#4) |
| 5 | [Ajuste sobre el entrenamiento](#5) |
| 6 | [Verificaciones](#6) |
| 7 | [Guardado del artefacto](#7) |

<a id="0"></a>
## 0. Configuración

In [1]:
from pathlib import Path
import time

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, PowerTransformer, StandardScaler

from predict_future_sales.config import MODELS_DIR, RAW_DATA_DIR
from predict_future_sales.features import (
    CaracteristicasCalendario,
    LagFeatures,
    Mapper,
    OutlierCapper,
    ReducirPrecision,
    UnirCatalogos,
    construir_rejilla,
    extraer_ciudad,
    extraer_tipo_categoria,
    panel_mensual,
)

# Pide a scikit-learn que los pasos devuelvan DataFrames en vez de matrices sin
# nombre. Cuesta algo de memoria, pero permite ver en todo momento que columna
# es cada cosa: sin esto, el resultado son 29 columnas anonimas.
sklearn.set_config(transform_output="pandas")

MODELS_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)
print("scikit-learn", sklearn.__version__)

2026-11-20 02:02:42.089 -06:00 | INFO     | predict_future_sales.config:<module>:11 - PROJ_ROOT path is: C:\Users\dgarc\OneDrive\Escritorio\Maestria\4T_2026\Product_Mang\Clase_Cloud\predict_future_sales


scikit-learn 1.9.1


In [2]:
sales = pd.read_csv(RAW_DATA_DIR / "sales_train.csv", encoding="utf-8")
items = pd.read_csv(RAW_DATA_DIR / "items.csv", encoding="utf-8")
cats = pd.read_csv(RAW_DATA_DIR / "item_categories.csv", encoding="utf-8")
shops = pd.read_csv(RAW_DATA_DIR / "shops.csv", encoding="utf-8")
print(f"{len(sales):,} ventas diarias cargadas")

2,935,849 ventas diarias cargadas


<a id="1"></a>
## 1. Los datos que el pipeline lleva dentro

Un pipeline normal solo guarda números aprendidos: medias, medianas, límites. Este además tiene
que guardar **tablas**, y la razón es el problema de los rezagos.

> `test.csv` corresponde al mes 34 y solo trae tienda y producto. Sus rezagos están en los meses
> 33, 32 y 31, que viven en los datos de **entrenamiento**. Si el pipeline no se los lleva dentro,
> el notebook de inferencia no puede calcularlos.

Por eso el artefacto incorpora tres tablas de apoyo.

In [3]:
# (a) Historial mensual por par tienda-producto: la serie que se rezaga.
panel = panel_mensual(sales)

# (b) Precio medio de cada producto en cada mes.
precios = (sales.groupby(["date_block_num", "item_id"], observed=True)["item_price"]
           .mean().rename("precio_medio").reset_index()
           .astype({"date_block_num": "int8", "item_id": "int16", "precio_medio": "float32"}))

tabla = pd.DataFrame([
    ("panel mensual", len(panel), panel.memory_usage(deep=True).sum() / 1e6),
    ("precios por mes", len(precios), precios.memory_usage(deep=True).sum() / 1e6),
    ("catalogo de productos", len(items), items[["item_id", "item_category_id"]]
     .memory_usage(deep=True).sum() / 1e6),
    ("catalogo de tiendas", len(shops), shops.memory_usage(deep=True).sum() / 1e6),
    ("catalogo de categorias", len(cats), cats.memory_usage(deep=True).sum() / 1e6),
], columns=["tabla", "filas", "MB"])
display(tabla.set_index("tabla").round(2))
print(f"total a guardar dentro del artefacto: {tabla.MB.sum():.1f} MB")

,filas,MB
tabla,,
panel mensual,1609124,12.87
precios por mes,233912,1.64
catalogo de productos,22170,0.35
catalogo de tiendas,60,0.01
catalogo de categorias,84,0.01


total a guardar dentro del artefacto: 14.9 MB


**Unos 20 MB en total.** Es un precio muy bajo por tener un artefacto autosuficiente: sin estas
tablas, el notebook 05 tendría que volver a leer `sales_train.csv` y repetir los cálculos, que es
justo el tipo de duplicación que produce errores difíciles de detectar.

### Una aclaración sobre fuga de información

El panel que se guarda contiene **los 34 meses**, incluidos los de validación. Esto puede parecer
fuga, y conviene explicar por qué no lo es.

El rezago *k* de una fila del mes *m* se busca con la clave `(tienda, producto, m − k)`. Para una
fila de entrenamiento del mes 20, el rezago 1 lee el mes 19. **Nunca** lee el mes 20 ni los
posteriores, aunque estén presentes en la tabla.

Y es exactamente la situación de la competencia: para predecir el mes 34 se permite usar lo que
realmente ocurrió en el mes 33, porque ya pasó. Lo que estaría prohibido es usar el mes 34 para
predecir el mes 34, y eso la clave del cruce lo hace imposible.

<a id="2"></a>
## 2. Los operadores que generan variables

Cuatro pasos, en orden. Cada uno **añade** columnas; ninguno elimina filas.

In [4]:
generadores = [
    ("catalogos", UnirCatalogos(items, shops, cats, mes_por_defecto=34)),
    ("calendario", CaracteristicasCalendario()),
    ("rezagos", LagFeatures(panel=panel, lags=(1, 2, 3, 12),
                            ventanas=(3, 6), precios=precios)),
    ("mapeos", Mapper({"ciudad": ("shop_name", extraer_ciudad),
                       "tipo_cat": ("item_category_name", extraer_tipo_categoria)})),
]

for nombre, op in generadores:
    print(f"{nombre:<12} -> {type(op).__name__}")
    print(f"             {(op.__doc__ or '').strip().splitlines()[0]}")

catalogos    -> UnirCatalogos
             Pega a cada fila los datos de catalogo: categoria del producto y nombres.
calendario   -> CaracteristicasCalendario
             Deriva del numero de mes todo lo que el calendario puede aportar.
rezagos      -> LagFeatures
             Crea los rezagos: cuanto vendio esta misma combinacion en meses anteriores.
mapeos       -> Mapper
             Traduce texto a codigos usando diccionarios aprendidos del entrenamiento.


| Paso | Qué añade | Por qué |
|---|---|---|
| `UnirCatalogos` | `item_category_id`, `shop_name`, `item_category_name` | Para el 48 % de pares sin historial, la categoría es casi lo único que informa |
| `CaracteristicasCalendario` | `mes_anio`, `anio`, `es_diciembre`, `dias_del_mes`, `findes_del_mes` | El EDA midió que diciembre es el único mes por encima del nivel, y que el sábado vende 1.82 veces lo del lunes |
| `LagFeatures` | 10 variables de historial | El rezago 1 es la variable más informativa (r = +0.361) |
| `Mapper` | `ciudad`, `tipo_cat` | Agrupa 60 tiendas en 31 ciudades y 84 categorías en 20 tipos |

**El `Mapper` se escribió para este proyecto.** El enunciado permite usar el operador visto en
clase, pero no estaba disponible, así que se implementó uno propio siguiendo la misma convención
que los operadores del curso (`BaseEstimator` + `TransformerMixin`, lo aprendido en atributos
terminados en guion bajo). Hace dos cosas que `OrdinalEncoder` no puede hacer solo: **extraer**
la ciudad de dentro del nombre de la tienda, y **marcar como nulo** —en vez de inventar un
código— cualquier valor que no estuviera en el entrenamiento.

<a id="3"></a>
## 3. El `ColumnTransformer`: los seis procesos

No todas las columnas necesitan lo mismo. Un `ColumnTransformer` aplica una secuencia distinta a
cada grupo y luego pega los resultados.

In [5]:
# Los cuatro bloques de columnas, tal como los definio el notebook 02.
COLS_REZAGO = ["lag_1", "lag_2", "lag_3", "lag_12", "media_3", "media_6",
               "item_lag_1", "shop_lag_1", "meses_de_vida"]
COLS_PRECIO = ["precio_lag_1"]
COLS_CALENDARIO = ["mes_anio", "anio", "es_diciembre", "dias_del_mes", "findes_del_mes"]
COLS_CATEGORICAS = ["shop_id", "item_id", "item_category_id", "ciudad", "tipo_cat"]

print(f"rezagos     : {len(COLS_REZAGO)} columnas")
print(f"precio      : {len(COLS_PRECIO)}")
print(f"calendario  : {len(COLS_CALENDARIO)}")
print(f"categoricas : {len(COLS_CATEGORICAS)}  <- las exogenas que pide el enunciado")

rezagos     : 9 columnas
precio      : 1
calendario  : 5
categoricas : 5  <- las exogenas que pide el enunciado


In [6]:
preproceso = ColumnTransformer(
    transformers=[
        # --- Bloque 1: los rezagos -------------------------------------
        # Imputacion con CERO porque un par sin historial no ha vendido: cero
        # es el valor correcto, no un relleno de conveniencia. `add_indicator`
        # anade ademas una bandera por columna que dice "este cero era en
        # realidad un dato ausente", para que el modelo pueda distinguir
        # "no vendio" de "no se sabe".
        ("rezagos", Pipeline([
            ("imputar", SimpleImputer(strategy="constant", fill_value=0, add_indicator=True)),
            ("escalar", StandardScaler()),
        ]), COLS_REZAGO),

        # --- Bloque 2: el precio ---------------------------------------
        # Aqui si tiene sentido la mediana (un precio de cero seria absurdo),
        # el recorte de extremos (el EDA encontro un precio de -1) y la
        # transformacion de sesgo (va de 5 a 307 980 rublos).
        ("precio", Pipeline([
            ("imputar", SimpleImputer(strategy="median")),
            ("recortar", OutlierCapper(inferior=0.01, superior=0.99)),
            ("yeojohnson", PowerTransformer(method="yeo-johnson")),
        ]), COLS_PRECIO),

        # --- Bloque 3: el calendario -----------------------------------
        # No hay nada que imputar: el mes de una fila siempre se conoce.
        ("calendario", StandardScaler(), COLS_CALENDARIO),

        # --- Bloque 4: las categoricas ---------------------------------
        # La moda es la unica imputacion con sentido para una categoria.
        # `unknown_value=-1` evita que el pipeline falle si en produccion
        # aparece una tienda o un producto que no existia al entrenar.
        ("categoricas", Pipeline([
            ("imputar", SimpleImputer(strategy="most_frequent")),
            ("codificar", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
        ]), COLS_CATEGORICAS),
    ],
    remainder="drop",              # lo que no se nombra, se descarta
    verbose_feature_names_out=False,  # nombres limpios, sin el prefijo del bloque
)
print(preproceso)

ColumnTransformer(transformers=[('rezagos',
                                 Pipeline(steps=[('imputar',
                                                  SimpleImputer(add_indicator=True,
                                                                fill_value=0,
                                                                strategy='constant')),
                                                 ('escalar',
                                                  StandardScaler())]),
                                 ['lag_1', 'lag_2', 'lag_3', 'lag_12',
                                  'media_3', 'media_6', 'item_lag_1',
                                  'shop_lag_1', 'meses_de_vida']),
                                ('precio',
                                 Pipeline(steps=[('imputar',
                                                  SimpleImputer(strategy='median')),
                                                 ('recortar',...
                                ('calendario', Stand

### Por qué codificación ordinal y no *one-hot*

Es la decisión de diseño con consecuencias más tangibles, y es puramente aritmética:

| Variable | Valores distintos | Columnas con *one-hot* | Memoria sobre 10.9 M filas |
|---|---|---|---|
| `item_id` | 21 807 | 21 807 | **950 GB** |
| `shop_id` | 60 | 60 | 2.6 GB |

La máquina tiene 8 GB. *One-hot* sobre `item_id` no es una opción discutible: no cabe.

La codificación ordinal ocupa **una columna** por variable. Su inconveniente conocido es que
introduce un orden que no existe —como si el producto 500 fuera "mayor" que el 200—, pero eso
**no afecta a los modelos de árbol**, que son los que mejor funcionan en este problema: un árbol
solo pregunta "¿es el código menor que X?" y con cortes sucesivos puede aislar cualquier grupo.

Para los modelos lineales, que sí sufrirían ese orden falso, la información categórica útil llega
por otra vía: las agrupaciones de baja cardinalidad que produce el `Mapper` (31 ciudades, 20 tipos
de categoría).

<a id="4"></a>
## 4. El pipeline ensamblado

In [7]:
pipeline_features = Pipeline([
    # --- Generacion de variables ---
    ("catalogos", UnirCatalogos(items, shops, cats, mes_por_defecto=34)),
    ("calendario", CaracteristicasCalendario()),
    ("rezagos", LagFeatures(panel=panel, lags=(1, 2, 3, 12),
                            ventanas=(3, 6), precios=precios)),
    ("mapeos", Mapper({"ciudad": ("shop_name", extraer_ciudad),
                       "tipo_cat": ("item_category_name", extraer_tipo_categoria)})),

    # Reduce a float32 ANTES del preprocesamiento. Sin esto, scikit-learn
    # trabaja en float64 y la matriz pasa de 1.1 GB a 2.2 GB, que no cabe
    # holgadamente en esta maquina.
    ("a_float32", ReducirPrecision()),

    # --- Los seis procesos de ingenieria ---
    ("preproceso", preproceso),

    # El escalado devuelve float64; se vuelve a bajar a float32.
    ("float32", ReducirPrecision()),
])

# IMPORTANTE: `sklearn.set_config(...)` de la celda inicial es una opcion
# GLOBAL del proceso, y no se guarda dentro del .pkl. Si el pipeline dependiera
# de ella, al abrirlo en otro proceso los pasos devolverian matrices sin nombre
# y el OutlierCapper fallaria al buscar sus columnas.
#
# `set_output` sobre el propio pipeline guarda la preferencia DENTRO del objeto,
# asi que viaja con el archivo. La diferencia solo se nota al cargar el
# artefacto desde fuera, que es justo lo que verifica la seccion 7.
pipeline_features.set_output(transform="pandas")

print("Pasos del pipeline:")
for i, (nombre, paso) in enumerate(pipeline_features.steps, 1):
    print(f"  {i}. {nombre:<12} {type(paso).__name__}")

Pasos del pipeline:
  1. catalogos    UnirCatalogos
  2. calendario   CaracteristicasCalendario
  3. rezagos      LagFeatures
  4. mapeos       Mapper
  5. a_float32    ReducirPrecision
  6. preproceso   ColumnTransformer
  7. float32      ReducirPrecision


<a id="5"></a>
## 5. Ajuste sobre el entrenamiento

El pipeline se ajusta **solo con los datos de entrenamiento**. El enunciado fija la partición:
el primer 80 % de los meses entrena y el 20 % final valida, de forma **secuencial y no aleatoria**,
porque en una serie temporal barajar significaría entrenar con el futuro para predecir el pasado.

Con 34 meses, eso son los **meses 0 a 26** para entrenar y los **27 a 33** para validar.

In [8]:
MES_CORTE = 26   # ultimo mes de entrenamiento (80 % de 34 meses)

rejilla = construir_rejilla(sales)
entrenamiento = rejilla[rejilla.date_block_num <= MES_CORTE]
validacion = rejilla[rejilla.date_block_num > MES_CORTE]

print(f"rejilla completa : {len(rejilla):,} filas")
print(f"entrenamiento    : {len(entrenamiento):,} filas  (meses 0-{MES_CORTE})")
print(f"validacion       : {len(validacion):,} filas  (meses {MES_CORTE + 1}-33)")
print(f"proporcion       : {len(entrenamiento) / len(rejilla):.0%} / "
      f"{len(validacion) / len(rejilla):.0%}")

rejilla completa : 10,913,850 filas
entrenamiento    : 9,299,486 filas  (meses 0-26)
validacion       : 1,614,364 filas  (meses 27-33)
proporcion       : 85% / 15%


In [9]:
# Las entradas son solo tres columnas; el objetivo va recortado a [0, 20]
# siguiendo la regla de la competencia.
ENTRADAS = ["date_block_num", "shop_id", "item_id"]
X_tr, y_tr = entrenamiento[ENTRADAS], entrenamiento["item_cnt_month"].clip(0, 20).astype("float32")
X_va, y_va = validacion[ENTRADAS], validacion["item_cnt_month"].clip(0, 20).astype("float32")

t0 = time.time()
Z_tr = pipeline_features.fit_transform(X_tr, y_tr)
print(f"ajustado y transformado en {time.time() - t0:.0f} s")
print(f"entrada: {X_tr.shape}  ->  salida: {Z_tr.shape}")
display(Z_tr.head(3))

ajustado y transformado en 72 s
entrada: (9299486, 3)  ->  salida: (9299486, 29)


,lag_1,lag_2,lag_3,lag_12,media_3,media_6,item_lag_1,shop_lag_1,meses_de_vida,missingindicator_lag_1,missingindicator_lag_2,missingindicator_lag_3,missingindicator_lag_12,missingindicator_media_3,missingindicator_media_6,missingindicator_item_lag_1,missingindicator_shop_lag_1,missingindicator_meses_de_vida,precio_lag_1,mes_anio,anio,es_diciembre,dias_del_mes,findes_del_mes,shop_id,item_id,item_category_id,ciudad,tipo_cat
0,-0.09986,-0.096836,-0.094494,-0.061814,-0.112998,-0.122116,-0.19924,-2.697186,-0.486289,4.946295,3.398758,2.6986,1.035244,4.946295,4.946295,4.946295,4.946295,-0.867763,0.082979,-1.41966,-0.93712,-0.289703,0.658654,-0.864524,58.0,19792.0,36.0,30.0,11.0
1,-0.09986,-0.096836,-0.094494,-0.061814,-0.112998,-0.122116,-0.19924,-2.697186,-0.486289,4.946295,3.398758,2.6986,1.035244,4.946295,4.946295,4.946295,4.946295,1.152389,0.082979,-1.41966,-0.93712,-0.289703,0.658654,-0.864524,58.0,2210.0,56.0,30.0,13.0
2,-0.09986,-0.096836,-0.094494,-0.061814,-0.112998,-0.122116,-0.19924,-2.697186,-0.486289,4.946295,3.398758,2.6986,1.035244,4.946295,4.946295,4.946295,4.946295,1.152389,0.082979,-1.41966,-0.93712,-0.289703,0.658654,-0.864524,58.0,2212.0,56.0,30.0,13.0


<a id="6"></a>
## 6. Verificaciones

Un pipeline que corre sin error no es un pipeline correcto. Se comprueban cinco cosas con
`assert`, de modo que si algo cambia en el futuro el notebook falle de forma ruidosa en lugar de
producir resultados silenciosamente equivocados.

In [10]:
# 1. Sin valores nulos: ningun modelo de scikit-learn los admite.
nulos = int(Z_tr.isna().sum().sum())
assert nulos == 0, f"Quedan {nulos} nulos"
print(f"1. nulos en la salida ................ {nulos}  OK")

# 2. Todo numerico y en float32.
tipos = set(map(str, Z_tr.dtypes))
assert tipos == {"float32"}, f"Tipos inesperados: {tipos}"
print(f"2. tipos de dato ..................... {tipos}  OK")

# 3. Nombres de columna legibles (no 'tmp_col_name_0_1').
assert not any(c.startswith("tmp_col_name") for c in Z_tr.columns)
print(f"3. columnas con nombre ............... {Z_tr.shape[1]} columnas  OK")

# 4. No se perdio ni se duplico ninguna fila.
assert len(Z_tr) == len(X_tr), "El pipeline cambio el numero de filas"
print(f"4. filas conservadas ................. {len(Z_tr):,}  OK")

# 5. Memoria dentro de lo previsto.
gb = Z_tr.memory_usage(deep=True).sum() / 1e9
assert gb < 2.0, f"La matriz ocupa {gb:.2f} GB"
print(f"5. memoria ........................... {gb:.2f} GB  OK")

1. nulos en la salida ................ 0  OK
2. tipos de dato ..................... {'float32'}  OK
3. columnas con nombre ............... 29 columnas  OK
4. filas conservadas ................. 9,299,486  OK
5. memoria ........................... 1.08 GB  OK


In [11]:
# El mismo pipeline, ya ajustado, se aplica a validacion SIN volver a aprender.
# Esta es la garantia de que no hay fuga: las medias, medianas y percentiles
# que se usan aqui salieron de los meses 0-26 y de ningun otro sitio.
t0 = time.time()
Z_va = pipeline_features.transform(X_va)
print(f"validacion transformada en {time.time() - t0:.0f} s -> {Z_va.shape}")

assert list(Z_va.columns) == list(Z_tr.columns), "Las columnas no coinciden"
assert int(Z_va.isna().sum().sum()) == 0
print("mismas columnas y en el mismo orden: OK")

validacion transformada en 8 s -> (1614364, 29)
mismas columnas y en el mismo orden: OK


In [12]:
# Comprobacion de que el escalado se aprendio SOLO con entrenamiento.
# Si se hubiera ajustado sobre todo, validacion tendria tambien media 0; al
# venir de otra epoca -y con una tendencia a la baja-, su media debe estar
# desplazada. Ver ese desplazamiento es la prueba de que no hubo fuga.
comparacion = pd.DataFrame({
    "media en entrenamiento": Z_tr[["lag_1", "lag_2", "media_3"]].mean(),
    "media en validacion": Z_va[["lag_1", "lag_2", "media_3"]].mean(),
}).round(4)
display(comparacion)
print("La media de entrenamiento es ~0 por construccion del escalado.")
print("Que la de validacion NO sea 0 es exactamente lo que se espera:")
print("son meses posteriores, con menos ventas, y el pipeline no los vio al ajustarse.")

,media en entrenamiento,media en validacion
lag_1,0.0,-0.0094
lag_2,-0.0,-0.0045
media_3,-0.0,-0.0082


La media de entrenamiento es ~0 por construccion del escalado.
Que la de validacion NO sea 0 es exactamente lo que se espera:
son meses posteriores, con menos ventas, y el pipeline no los vio al ajustarse.


In [13]:
# Prueba de que el mes 34 se puede transformar aunque no exista en los datos.
# Es el caso real del notebook 05: test.csv no trae date_block_num.
test = pd.read_csv(RAW_DATA_DIR / "test.csv", encoding="utf-8")
Z_test = pipeline_features.transform(test[["shop_id", "item_id"]])
print(f"test.csv transformado: {test.shape} -> {Z_test.shape}")
assert len(Z_test) == len(test)
assert int(Z_test.isna().sum().sum()) == 0
print("El pipeline rellena date_block_num = 34 y calcula los rezagos desde su historial: OK")

test.csv transformado: (214200, 3) -> (214200, 29)
El pipeline rellena date_block_num = 34 y calcula los rezagos desde su historial: OK


<a id="7"></a>
## 7. Guardado del artefacto

**Entregable 2 del proyecto.**

In [14]:
RUTA_PKL = MODELS_DIR / "pipeline_features.pkl"
joblib.dump(pipeline_features, RUTA_PKL, compress=3)
print(f"guardado en : {RUTA_PKL}")
print(f"tamanio     : {RUTA_PKL.stat().st_size / 1e6:.1f} MB")

guardado en : C:\Users\dgarc\OneDrive\Escritorio\Maestria\4T_2026\Product_Mang\Clase_Cloud\predict_future_sales\models\pipeline_features.pkl
tamanio     : 8.2 MB


### La prueba que de verdad importa

Guardar el archivo no demuestra nada. Lo que hay que demostrar es que **se puede volver a abrir
desde otro proceso**, arrancado en otro directorio, y que produce el mismo resultado.

Es la comprobación que delata el error más común al serializar modelos: si los operadores propios
estuvieran definidos dentro del notebook, el `.pkl` guardaría solo una referencia a ellos y al
abrirlo desde fuera fallaría con `ModuleNotFoundError`. Por eso viven en el paquete.

**Esta prueba ya encontró un defecto real.** En la primera versión, el pipeline funcionaba
perfectamente dentro del notebook y fallaba al cargarlo desde otro proceso, con un
`KeyError: 'precio_lag_1'`. La causa: `sklearn.set_config(transform_output="pandas")` es una
opción **global del proceso**, no una propiedad del objeto, así que no viajaba dentro del
archivo. Fuera del notebook, los pasos devolvían matrices sin nombres de columna y el recorte de
extremos no encontraba la suya. Se resolvió con `pipeline_features.set_output(...)`, que sí
guarda la preferencia dentro del objeto.

Sin esta verificación, el defecto habría aparecido en el notebook 05 o, peor, en quien recibiera
el artefacto.

In [15]:
import subprocess
import sys
import tempfile
import textwrap

# Se escribe un script que carga el .pkl y predice, y se ejecuta desde un
# directorio TEMPORAL, fuera del proyecto. Si funciona ahi, funciona en
# cualquier sitio.
guion = textwrap.dedent(f"""
    import joblib, pandas as pd
    pipe = joblib.load(r"{RUTA_PKL}")
    muestra = pd.DataFrame({{"shop_id": [5, 25], "item_id": [5037, 2808]}})
    salida = pipe.transform(muestra)
    print("CARGA OK |", salida.shape, "| columnas:", len(salida.columns))
    print("primera fila:", salida.iloc[0, :4].round(3).to_dict())
""")
with tempfile.TemporaryDirectory() as carpeta:
    script = Path(carpeta) / "prueba_carga.py"
    script.write_text(guion, encoding="utf-8")
    r = subprocess.run([sys.executable, str(script)], capture_output=True,
                       text=True, cwd=carpeta)
print("directorio de ejecucion: una carpeta temporal, fuera del proyecto")
print(r.stdout or r.stderr[-800:])
assert "CARGA OK" in r.stdout, "El artefacto no se puede cargar desde fuera del proyecto"

directorio de ejecucion: una carpeta temporal, fuera del proyecto
CARGA OK | (2, 29) | columnas: 29
primera fila: {'lag_1': -0.10000000149011612, 'lag_2': 0.22100000083446503, 'lag_3': 0.8899999856948853, 'lag_12': 0.3720000088214874}



---

## Resumen

| | |
|---|---|
| **Artefacto** | `models/pipeline_features.pkl` |
| **Entrada** | `shop_id`, `item_id` (+ `date_block_num` opcional) |
| **Salida** | 29 columnas `float32`, sin nulos, escaladas |
| **Ajustado con** | Meses 0 a 26 (9 299 486 filas) |
| **Operadores propios** | `UnirCatalogos`, `CaracteristicasCalendario`, `LagFeatures`, `Mapper`, `OutlierCapper`, `ReducirPrecision` |
| **Operadores de scikit-learn** | `SimpleImputer` ×3, `StandardScaler` ×2, `PowerTransformer`, `OrdinalEncoder`, `ColumnTransformer` |
| **Verificado** | Sin nulos · tipos correctos · filas conservadas · se carga desde otro proceso |

**Siguiente paso:** el notebook `04_model_training_n_tunning.ipynb` enfrenta 15 modelos sobre
estas variables y añade el ganador como paso final del pipeline.